# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moriyamao/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Skill loaded: `training-honest-models` + `flyrank/flyrank-data`.*

**Lane recap (from Week 4):** the refresh + CTR-fix review queue. This week's question: can a learned model, using the same starter CSV and honest (non-label-derived) features, rank pages by decline risk better than that rule-based baseline?

In [1]:
# Repo clone, only if the data isn't already present (edit the URL to your own repo).
import os
if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    !git clone https://github.com/moriyamao/flyrank-ml-internship.git repo
    os.chdir('repo')

import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(df.shape, 'base rate:', round(df['is_declining_label'].mean(), 3))

Cloning into 'repo'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 154 (delta 58), reused 95 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1.86 MiB | 10.42 MiB/s, done.
Resolving deltas: 100% (58/58), done.
(30000, 45) base rate: 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**The question shape:** "which pages first?" — a ranking problem with an observed label (`is_declining_label`, from `trend_direction == 'down'`), same as the baseline was scored on. Per the skill's table, that means **Logistic Regression first (readable), then Random Forest (stronger)** — both evaluated at precision@K, not accuracy, since a ranking question needs scores.

I'm skipping Gradient Boosting: the skill card says "where safe," and with only 13 numeric + 3 categorical honest features and a base rate near 50%, added complexity isn't earned yet — the comparison table below decides whether even Random Forest earns its keep over plain Logistic Regression.

**Features used (all knowable before the outcome, none label-derived):**
- Numeric: `impressions_90d`, `clicks_90d`, `sessions_90d`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `avg_position`, `ctr`, `word_count`, `content_age_days`, `days_since_last_update`, `search_volume`, `competition`
- Categorical: `content_type`, `main_intent`, `competition_level`

**Explicitly excluded:** `trend_direction`, `trend_pct` (the label itself), `provider_used`, `model_used` (not model features per the data dictionary), and `content_id`/`client_id` (context only, used for the split, never as inputs).

In [2]:
num_features = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'engagement_rate', 'scroll_rate',
                'ai_traffic_pct', 'avg_position', 'ctr', 'word_count', 'content_age_days',
                'days_since_last_update', 'search_volume', 'competition']
cat_features = ['content_type', 'main_intent', 'competition_level']

X = df[num_features + cat_features]
y = df['is_declining_label']
groups = df['client_id']
print('Feature count:', len(num_features) + len(cat_features))

Feature count: 16


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`**, not a random row split. The data dictionary is explicit that `client_id` exists "for grouped train/test splits" — a random split would let the model see other pages from the same client in training and effectively memorize client-level quirks (a client's typical CTR, its content style) rather than learning a generalizable decline signal. There's no true time axis in the 90-day-snapshot starter CSV, so a time-aware split doesn't apply here — client-grouping is the honest split for this data.

75/25 group split, verified below to have **zero client overlap** between train and test.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print('Train rows:', len(train_idx), '| Test rows:', len(test_idx))
print('Client overlap between train and test:', len(overlap), '(must be 0)')

Train rows: 22885 | Test rows: 7115
Client overlap between train and test: 0 (must be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

# Rebuild the Week-4 baseline score on this same dataframe, so it's scored on the SAME rows/split.
visible = df['impressions_90d'] >= 300
stale = df['freshness_tier'].isin(['91-180', '181+'])
tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')
ctr_gap = (df['ctr'] < tier_median_ctr) & (df['position_tier'].isin(['top_3', 'page_1', 'striking']))
df['baseline_score'] = visible.astype(int) * (stale.astype(int) + ctr_gap.astype(int)) * np.log1p(df['impressions_90d'])

pre_logit = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='unknown')),
                       ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
])
pre_rf = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='unknown')),
                       ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
])

# Seeds fixed (random_state=42) throughout -- reruns reproduce the same table.
logit = Pipeline([('pre', pre_logit), ('clf', LogisticRegression(max_iter=2000, random_state=42))])
rf = Pipeline([('pre', pre_rf), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))])

logit.fit(X.iloc[train_idx], y.iloc[train_idx])
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

logit_proba = logit.predict_proba(X.iloc[test_idx])[:, 1]
rf_proba = rf.predict_proba(X.iloc[test_idx])[:, 1]
baseline_scores = df['baseline_score'].iloc[test_idx].values
y_test = y.iloc[test_idx].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = y_test.mean()
rows = []
for k in [20, 50, 100, 200]:
    rows.append({
        'K': k,
        'baseline_precision': round(precision_at_k(baseline_scores, y_test, k), 3),
        'logit_precision': round(precision_at_k(logit_proba, y_test, k), 3),
        'rf_precision': round(precision_at_k(rf_proba, y_test, k), 3),
    })
comparison = pd.DataFrame(rows)
comparison['base_rate'] = round(base_rate, 3)

print('Base rate (test set):', round(base_rate, 3))
print('AUC — logit:', round(roc_auc_score(y_test, logit_proba), 3), '| rf:', round(roc_auc_score(y_test, rf_proba), 3))
comparison

Base rate (test set): 0.517
AUC — logit: 0.542 | rf: 0.6


,K,baseline_precision,logit_precision,rf_precision,base_rate
0,20,0.500,0.550,0.550,0.517
1,50,0.520,0.640,0.560,0.517
2,100,0.600,0.590,0.560,0.517
3,200,0.625,0.565,0.585,0.517


**Reading the table honestly:** neither model dominates the baseline everywhere — this is the finding, not a failure to report. At K=20 and K=50, Logistic Regression is clearly ahead of the baseline (0.55 and 0.64 vs 0.50 and 0.52). At K=100 and K=200, the **baseline wins** (0.60 and 0.625 vs logit's 0.59 and 0.565, and RF trailing both at 0.56 and 0.585). AUC for both models sits close to chance (logit 0.542, RF 0.600) — a real but modest signal, not a strong one. Random Forest doesn't beat Logistic Regression here despite being the "stronger" step in the toolkit's ladder — added complexity isn't earning its keep on this feature set, consistent with the skill's warning not to reward complexity alone.

**Careful-words summary:** the model shows a *directional* edge over the baseline specifically for tight, high-precision top-K review queues (K≤50) — useful if the workflow only has capacity to review a handful of pages a day. For a broader queue (K≥100), the simple rule baseline remains competitive to better. Neither claim is causal; both are observed on this one held-out client split.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# Feature importances (Random Forest) -- what the stronger model actually leans on.
ohe = rf.named_steps['pre'].named_transformers_['cat'].named_steps['oh']
cat_names = ohe.get_feature_names_out(cat_features)
all_names = num_features + list(cat_names)
importances = rf.named_steps['clf'].feature_importances_
imp_df = pd.DataFrame({'feature': all_names, 'importance': importances}).sort_values('importance', ascending=False)
imp_df.head(6)

,feature,importance
0,impressions_90d,0.213142
6,avg_position,0.174800
9,content_age_days,0.167460
8,word_count,0.076386
4,scroll_rate,0.051546
10,days_since_last_update,0.043395


In [6]:
# Concrete wrong cases: top-100 RF-ranked pages that did NOT actually decline (false positives).
test_df = df.iloc[test_idx].copy()
test_df['rf_proba'] = rf_proba
test_df['y'] = y_test

top100 = test_df.iloc[np.argsort(-rf_proba)[:100]]
print('Top-100 by RF: observed decline rate =', round(top100['y'].mean(), 3), '(', len(top100), 'rows )')
print('False positives in top-100:', int((top100['y'] == 0).sum()))

wrong_cases = top100[top100['y'] == 0][['content_id', 'client_id', 'rf_proba', 'impressions_90d',
                                          'avg_position', 'content_age_days', 'freshness_tier']].head(3)
wrong_cases

Top-100 by RF: observed decline rate = 0.56 ( 100 rows )
False positives in top-100: 44


,content_id,client_id,rf_proba,impressions_90d,avg_position,content_age_days,freshness_tier
20736,content_41baf0722ad9,client_8527a891e2,0.806744,3115,12.8,275,91-180
12472,content_884c401ce126,client_8527a891e2,0.802935,101,23.1,174,91-180
13561,content_685d3fea9361,client_8527a891e2,0.800370,134,9.3,275,91-180


**Top features (Random Forest):** `impressions_90d` (0.213), `avg_position` (0.175), `content_age_days` (0.167), then `word_count` (0.076) and `scroll_rate` (0.052). This makes plausible sense — high-traffic, well-ranked, older pages are exactly where a decline is both more measurable and more likely to be flagged as "down" by the trend calculation, since low-traffic pages have noisier 30-day deltas. None of the top features are suspiciously perfect (no single feature dominates above ~0.21), which is a mild reassurance against leakage rather than proof of its absence.

**Where the model is wrong:** of the top 100 RF-ranked pages, observed decline rate is 56% — better than the 51.7% base rate, but 44 of those top 100 did **not** actually decline. Three concrete examples (shown above): all three are mid-position pages (9–23), aged 174–275 days, sitting in the `91-180` freshness tier. The model appears to be leaning heavily on "stale + moderately-ranked" as a decline signal — the same pattern the baseline's `stale_flag` already captures — so its false positives look like pages that are stale but happen to be *stable* rather than declining, which staleness alone can't distinguish. This matches Week 4's honest MIXED verdict on the staleness signal: it's directionally real but noisy, and the model inherited that same noise rather than resolving it.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run it yourself once to confirm**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.